In [ ]:
import os

# Vast.ai HF credential location
os.environ["HF_HOME"] = "/workspace/.hf_home"

from huggingface_hub import login, get_token

hf_token = get_token()

if hf_token:
    print("Using cached Hugging Face authentication.")
else:
    raise RuntimeError("Hugging Face token not found.")

Hugging Face token not found. Add it in Colab Secrets.


In [ ]:
# Install specific compatible versions
import sys

!{sys.executable} -m pip install --upgrade pip setuptools wheel
!{sys.executable} -m pip install numpy scikit-learn


!{sys.executable} -m pip install git+https://github.com/AIPHES/emnlp19-moverscore.git

!{sys.executable} -m pip install accelerate rouge-score bert-score nltk torch torchvision pyemd datasets transformers peft "torchao>=0.16.0"

import nltk
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
nltk.download('punkt', quiet=True)

# Verify
try:
    from moverscore_v2 import get_idf_dict, word_mover_score
    print("✓ MoverScore installed successfully")
except Exception as e:
    print(f"✗ Error: {e}")

  Using cached setuptools-84.0.0-py3-none-any.whl.metadata (6.6 kB)
Using cached setuptools-84.0.0-py3-none-any.whl (818 kB)
  Attempting uninstall: setuptools
    Found existing installation: setuptools 81.0.0
    Uninstalling setuptools-81.0.0:
      Successfully uninstalled setuptools-81.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.11.0+cu128 requires setuptools<82, but you have setuptools 84.0.0 which is incompatible.
  Cloning https://github.com/AIPHES/emnlp19-moverscore.git to ./tmp/pip-req-build-rcaxd3sw
  Running command git clone --filter=blob:none --quiet https://github.com/AIPHES/emnlp19-moverscore.git /tmp/pip-req-build-rcaxd3sw
  Resolved https://github.com/AIPHES/emnlp19-moverscore.git to commit 0459a3b3b3bd73baa0cc515a355228ee5a2887e0
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparin

Failed to load /venv/main/lib/python3.12/site-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /venv/main/lib/python3.12/site-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /venv/main/lib/python3.12/site-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /venv/main/lib/python3.12/site-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ MoverScore installed successfully


In [ ]:
import nltk; nltk.download('wordnet', quiet=True); nltk.download('omw-1.4', quiet=True)

import torch
import gc
import math
import json
import os
import re
import threading
from typing import Dict, Optional, Tuple
from PIL import Image
from io import BytesIO
import requests

# Patch for numpy.float deprecation in MoverScore
import numpy as np
if not hasattr(np, 'float'):
    np.float = float
SEED = 42

# ---------------------------------------------------------------------------
# RUN MODE
#   SMOKE_TEST = True  -> 32 / 4 / 4 examples (24 optimizer steps)
#   SMOKE_TEST = False -> 8000 / 1000 / 1000 full protocol
# ---------------------------------------------------------------------------
SMOKE_TEST = False

RUN_NAME = f"LoRA_llama4_seed{SEED}_VG" + ("_smoke" if SMOKE_TEST else "")
OUTPUT_DIR = os.path.abspath(RUN_NAME)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"All experiment outputs will be saved in:\n{OUTPUT_DIR}")

if "hf_token" not in globals():
    hf_token = os.getenv("HF_TOKEN")

from transformers import EvalPrediction
nltk.download('punkt', quiet=True)

from datasets import load_dataset
from transformers import (
    MllamaForConditionalGeneration,
    AutoProcessor,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    set_seed,
)
from moverscore_v2 import get_idf_dict, word_mover_score
from peft import LoraConfig, get_peft_model


# Instruction used for BOTH train and eval: lmms-eval's base RefCOCO
# instruction + our brevity suffix (identical to the Gemma run).
REFCOCO_INSTRUCTION = (
    "Provide a short description for this region. "
    "Respond with only the short description itself "
    "(a few words), with no explanation or extra text."
)

# Keys the collator puts in the batch that are metadata, NOT model inputs.
# Gemma's forward silently swallowed unknown kwargs; Mllama forwards **kwargs
# down into the text model, so these must be stripped before every forward.
NON_MODEL_KEYS = ("refcoco_answers", "prompt_length")


def load_image_from_source(image_source):
    """
    Robust image loading for RefCOCO examples.

    lmms-lab-encoder/RefCOCO's `image` column is already a decoded
    PIL.Image (the raw COCO image, with NO region visualization drawn on
    it — see draw_region_indicator() below for that), so this is mostly a
    defensive convert-to-RGB helper.
    """
    try:
        if isinstance(image_source, Image.Image):
            return image_source.convert('RGB') if image_source.mode != 'RGB' else image_source

        # Fallback in case a path/bytes slips through instead of a PIL Image.
        if isinstance(image_source, str) and os.path.exists(image_source):
            image = Image.open(image_source)
            return image.convert('RGB') if image.mode != 'RGB' else image

        if isinstance(image_source, bytes):
            from io import BytesIO
            image = Image.open(BytesIO(image_source))
            return image.convert('RGB') if image.mode != 'RGB' else image

        return None

    except Exception as e:
        print(f"Failed to load image: {e}")
        return None


def draw_region_indicator(image, bbox, color=(255, 0, 0), width_frac=0.006):
    """
    Draw a red RECTANGLE around the referred region.

    FIX: this used to draw a padded ellipse and the prompt referred to
    "the area circled in the image". That doesn't match how this dataset
    is actually meant to be used: lmms-eval's own reference implementation
    for this exact dataset (lmms_eval/tasks/refcoco/utils.py,
    refcoco_bbox_doc_to_visual) draws a plain rectangle at the *exact*
    bbox, no padding, no circle:

        bbox_xy = [bbox[0], bbox[1], bbox[0] + bbox[2], bbox[1] + bbox[3]]
        draw.rectangle(bbox_xy, outline="red")

    We mirror that (bbox is COCO-style [x, y, w, h] in pixel coordinates,
    confirmed against the dataset's own schema) so the visual indicator
    matches what the harness/paper convention actually is, and so the
    prompt's wording ("the region" / "the box", not "circled") matches
    what's drawn. We keep a slightly thicker line than the harness's
    default 1px, since a 1px line is prone to disappearing once the image
    gets downsampled by the vision processor before the model ever sees it
    — the harness only ever runs zero-shot inference on the original-size
    image, so it doesn't have to worry about that, but we do.
    """
    from PIL import ImageDraw

    annotated = image.copy()
    if not bbox or len(bbox) != 4:
        return annotated

    x, y, w, h = bbox
    if w <= 0 or h <= 0:
        return annotated

    # Origin format (top x, top y, width, height) -> (x0, y0, x1, y1),
    # exactly as lmms-eval's refcoco_bbox_doc_to_visual does it. No padding.
    x0, y0 = x, y
    x1 = min(annotated.width, x + w)
    y1 = min(annotated.height, y + h)

    line_width = max(2, int(round(min(annotated.width, annotated.height) * width_frac)))

    draw = ImageDraw.Draw(annotated)
    draw.rectangle([x0, y0, x1, y1], outline=color, width=line_width)
    return annotated


def extract_raw_answers(example):
    """
    Return the list of raw human reference-expression strings for a
    RefCOCO example, for use with the soft-accuracy metric.

    Unlike VQAv2's dict-based `answers` field, RefCOCO's `answer` field is
    already a flat list of plain strings (the 2-4 human referring
    expressions collected for that region).
    """
    answers_field = example.get('answer')
    if not answers_field:
        return []
    raw = []
    for a in answers_field:
        ans = a.get('answer') if isinstance(a, dict) else a
        if ans:
            raw.append(str(ans))
    return raw


class LoRAConfig:
    """Configuration for LoRA-only training on Llama-3.2-11B-Vision-Instruct (RefCOCO)."""

    def __init__(self):
        self.model_id = "meta-llama/Llama-3.2-11B-Vision-Instruct"
        self.batch_size = 1
        self.gradient_accumulation_steps = 4
        self.num_epochs = 3
        self.precision = "bf16"
        # Mllama puts a single <|image|> token in the text stream (vision
        # enters through cross-attention), so prompt+answer is ~40-70 tokens.
        self.max_length = 512

        # LoRA configuration (identical to the Llama GRIT / Gated GRIT runs)
        self.lora_rank = 4
        self.lora_alpha = 16
        self.lora_dropout = 0.05

        # Language tower: self-attn AND cross-attn layers of the Llama text model
        self.lora_target_modules = [
            "q_proj",
            "k_proj",
            "v_proj",
        ]

        # Vision tower (Mllama vision encoder: local + global transformer)
        self.apply_lora_to_vision = True
        self.vision_lora_modules = [
            "q_proj",
            "k_proj",
            "v_proj",
        ]

        if SMOKE_TEST:
            self.train_size, self.val_size, self.test_size = 64, 8, 8
            self.learning_rate = 5e-5   # same as the GRIT / Gated GRIT smoke runs
            self.logging_steps = 1
            self.warmup_steps = 1
        else:
            # Full protocol = the values the Gemma LoRA RefCOCO run used.
            self.train_size, self.val_size, self.test_size = 8000, 1000, 1000
            self.learning_rate = 5e-5
            self.logging_steps = 50
            self.warmup_steps = 200

        # Dataset configuration
        # lmms-lab-encoder/RefCOCO is an eval-only repackaging (val/test/testA/testB,
        # no "train"). "val" -> train, "testA" -> validation, "testB" -> test.
        self.dataset_name = "lmms-lab-encoder/RefCOCO"
        self.num_workers = 0
        self.pin_memory = True
        self.drop_last = True

        self.answer_max_length = 512
        self.ignore_index = -100

        # Reference expressions are a few words; cap eval generation.
        self.generation_max_new_tokens = 64


class LazyRefCOCODataset(torch.utils.data.Dataset):
    """
    Tokenizes on the fly instead of datasets.map().

    Mllama pixel_values are [1 image, 4 tiles, 3, 560, 560] per example
    (~15 MB in fp32). Materializing them into Arrow via .map() is fine for
    32 examples but blows up disk / hits the Arrow serialization overflow at
    8000 (same failure we fixed before with SimpleListDataset). Only the
    `answer` column is scanned up front; images are processed per __getitem__.

    If an example fails to tokenize, the next valid one is returned instead.
    Every returned item carries its OWN refcoco_answers, so prediction <->
    reference alignment in the metrics is preserved.
    """

    def __init__(self, hf_ds, tokenize_fn):
        self.ds = hf_ds
        self.tokenize_fn = tokenize_fn
        answers = hf_ds["answer"]
        self.indices = [
            i for i, a in enumerate(answers)
            if a and str(a[0] if isinstance(a, (list, tuple)) else a).strip()
        ]

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        n = len(self.indices)
        for off in range(n):
            j = self.indices[(idx + off) % n]
            out = self.tokenize_fn(self.ds[j])
            if out is not None:
                if off > 0:
                    print(f"[LazyRefCOCODataset] example {self.indices[idx % n]} failed; substituted {j}")
                return out
        raise RuntimeError("No example in this split could be tokenized.")


def build_lora_target_regex(config):
    """
    Language tower: q/k/v/o on the Llama text model (self-attn + cross-attn layers).
    Vision tower  : q/k/v only on the Mllama vision encoder.
    A plain list ["q_proj","k_proj","v_proj","o_proj"] would ALSO hit the
    vision encoder's o_proj, so we scope each list to its tower with a regex
    (PEFT does re.fullmatch on module names). Matches both the old
    (language_model.model.layers...) and new (model.language_model.layers...)
    transformers module layouts.
    """
    lang = "|".join(re.escape(m) for m in config.lora_target_modules)
    parts = [rf".*language_model\..*\.({lang})"]
    if config.apply_lora_to_vision:
        vis = "|".join(re.escape(m) for m in config.vision_lora_modules)
        parts.append(rf".*vision_model\..*\.({vis})")
    return "|".join(f"(?:{p})" for p in parts)


def main():
    set_seed(SEED)
    config = LoRAConfig()

    print("Loading processor and model for Llama-3.2-11B-Vision-Instruct (RefCOCO)...")
    processor = AutoProcessor.from_pretrained(
        config.model_id,
        token=hf_token,
        trust_remote_code=True,
        padding_side="left",
    )
    tokenizer = processor.tokenizer

    if tokenizer.pad_token is None:
        # Llama-3 ships a dedicated pad token; fall back to EOS only if absent.
        if "<|finetune_right_pad_id|>" in tokenizer.get_vocab():
            tokenizer.pad_token = "<|finetune_right_pad_id|>"
        else:
            tokenizer.pad_token = tokenizer.eos_token
    print(f"Pad token: {tokenizer.pad_token!r} (id={tokenizer.pad_token_id})")

    # Stop generation on any Llama-3 end marker (instruct turns end with <|eot_id|>).
    eos_token_ids = []
    for t in [tokenizer.eos_token, "<|eot_id|>", "<|end_of_text|>"]:
        if t is None:
            continue
        tid = tokenizer.convert_tokens_to_ids(t)
        if isinstance(tid, int) and tid >= 0 and tid != tokenizer.unk_token_id and tid not in eos_token_ids:
            eos_token_ids.append(tid)
    print(f"EOS token ids for generation: {eos_token_ids}")

    model = MllamaForConditionalGeneration.from_pretrained(
        config.model_id,
        token=hf_token,
        torch_dtype=torch.bfloat16,
        attn_implementation="sdpa",
    )
    # NOTE: deliberately no device_map="auto" (meta-tensor issues in earlier Llama runs).

    target_regex = build_lora_target_regex(config)
    print(f"LoRA target regex: {target_regex}")

    lora_config = LoraConfig(
        r=config.lora_rank,
        lora_alpha=config.lora_alpha,
        target_modules=target_regex,
        lora_dropout=config.lora_dropout,
        bias="none",
        task_type="CAUSAL_LM",
        inference_mode=False,
        use_rslora=True,
    )

    model = get_peft_model(model, lora_config)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    meta_params = [name for name, param in model.named_parameters() if param.device.type == "meta"]
    if meta_params:
        raise RuntimeError(f"Model still has meta parameters: {meta_params[:10]}")
    print("✓ Model has no meta parameters")
    print(f"✓ Model device: {next(model.parameters()).device}")

    # Sanity: LoRA must land on BOTH towers.
    from peft.tuners.lora import LoraLayer
    lora_names = [n for n, m in model.named_modules() if isinstance(m, LoraLayer)]
    n_vis = sum('vision_model' in n for n in lora_names)
    n_lang = sum('language_model' in n for n in lora_names)
    print(f"Instrumented {n_vis} vision modules + {n_lang} language modules")
    print(f"Total LoRA modules: {n_vis + n_lang}")
    if n_lang == 0 or (config.apply_lora_to_vision and n_vis == 0):
        raise RuntimeError("LoRA target regex did not match both towers — check module names.")

    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())

    print(f"Trainable params: {trainable_params:,}")
    print(f"Total params: {total_params:,}")
    print(f"Percentage: {100 * trainable_params / total_params:.3f}%")

    model.enable_input_require_grads()
    model.config.use_cache = False
    if getattr(model.config, "text_config", None) is not None:
        model.config.text_config.use_cache = False

    print("Loading RefCOCO dataset with splits...")
    full_dataset = load_dataset(config.dataset_name)

    def _take(split, n):
        ds = full_dataset[split].shuffle(seed=SEED)
        return ds.select(range(min(n, len(ds))))

    train_dataset = _take("val", config.train_size)
    val_dataset   = _take("testA", config.val_size)
    test_dataset  = _take("testB", config.test_size)

    print(f"Final splits → Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")

    _debug_state = {"printed": False}

    def tokenize_refcoco(example):
        """
        Llama / Mllama multimodal tokenization.

          1) prompt_text = chat template of [user: <image> + instruction],
             add_generation_prompt=True  (ends with the assistant header).
          2) full_text   = chat template of [user ..., assistant: answer]
             (ends with answer + <|eot_id|>).
          3) ONE processor call on (image, full_text) -> input_ids,
             attention_mask, pixel_values, aspect_ratio_ids,
             aspect_ratio_mask, cross_attention_mask, all mutually consistent.
          4) prompt_length P = len(tokenized prompt_text). We ASSERT the
             prompt ids are an exact prefix of the full ids, then
             labels = -100 on [:P], answer ids (+ <|eot_id|>) on [P:].

        The chat template already emits <|begin_of_text|>, so the processor
        is called with add_special_tokens=False (no double BOS).
        We never manually concatenate answer ids onto processor tensors.
        """
        try:
            if 'image' not in example or example['image'] is None:
                return None

            image = load_image_from_source(example['image'])
            if image is None or not isinstance(image, Image.Image):
                return None
            if image.mode != 'RGB':
                image = image.convert('RGB')

            # Draw the referred region (red box at the GT bbox), as in the Gemma run.
            image = draw_region_indicator(image, example.get('bbox'))

            # Train on the first human expression; all 2-4 kept for eval metrics.
            answers_list = example.get('answer') or []
            answer = (answers_list[0] if answers_list else '').strip()
            if not answer:
                return None

            user_turn = {
                "role": "user",
                "content": [
                    {"type": "image"},
                    {"type": "text", "text": REFCOCO_INSTRUCTION},
                ],
            }
            assistant_turn = {
                "role": "assistant",
                "content": [{"type": "text", "text": answer}],
            }

            prompt_text = processor.apply_chat_template(
                [user_turn], add_generation_prompt=True, tokenize=False
            )
            full_text = processor.apply_chat_template(
                [user_turn, assistant_turn], add_generation_prompt=False, tokenize=False
            )

            if "<|image|>" not in prompt_text:
                print("ERROR: <|image|> token missing from chat template output.")
                return None

            full = processor(
                images=[[image]],
                text=[full_text],
                add_special_tokens=False,
                return_tensors="pt",
            )

            input_ids = full["input_ids"][0].long()
            attention_mask = full["attention_mask"][0].long()

            prompt_ids = tokenizer(
                prompt_text, add_special_tokens=False, return_tensors="pt"
            )["input_ids"][0].long()
            P = int(prompt_ids.shape[0])

            if P >= input_ids.shape[0] or not torch.equal(input_ids[:P], prompt_ids):
                print("WARNING: prompt ids are not a prefix of full ids — skipping example.")
                return None

            if input_ids.shape[0] > config.max_length:
                return None

            labels = input_ids.clone()
            labels[:P] = config.ignore_index

            cross_attention_mask = full["cross_attention_mask"][0]  # [seq, n_images, n_tiles]
            if cross_attention_mask.shape[0] != input_ids.shape[0]:
                print("WARNING: cross_attention_mask / input_ids length mismatch — skipping example.")
                return None

            output = {
                "input_ids": input_ids,
                "attention_mask": attention_mask,
                "labels": labels,
                "cross_attention_mask": cross_attention_mask,
                "pixel_values": full["pixel_values"][0],            # [n_images, n_tiles, 3, 560, 560]
                "aspect_ratio_ids": full["aspect_ratio_ids"][0],    # [n_images]
                "aspect_ratio_mask": full["aspect_ratio_mask"][0],  # [n_images, n_tiles]
                # metadata (stripped before the model sees the batch)
                "prompt_length": P,
                "refcoco_answers": extract_raw_answers(example),
            }

            if not _debug_state["printed"]:
                _debug_state["printed"] = True
                tgt = input_ids[P:]
                print("\n[Tokenization Debug] ---------------------------------")
                print(f"Full seq len: {input_ids.shape[0]} | prompt len: {P} | target len: {tgt.shape[0]}")
                print(f"Prompt tail : {tokenizer.decode(input_ids[max(0, P - 8):P])!r}")
                print(f"Target (raw): {tokenizer.decode(tgt)!r}")
                print(f"pixel_values {tuple(output['pixel_values'].shape)} | "
                      f"aspect_ratio_ids {tuple(output['aspect_ratio_ids'].shape)} | "
                      f"aspect_ratio_mask {tuple(output['aspect_ratio_mask'].shape)} | "
                      f"cross_attention_mask {tuple(cross_attention_mask.shape)}")
                print("-----------------------------------------------------\n")

            return output

        except Exception as e:
            print(f"Error processing example: {e}")
            import traceback
            traceback.print_exc()
            return None

    print("Building lazy datasets (on-the-fly tokenization)...")
    tokenized_train_dataset = LazyRefCOCODataset(train_dataset, tokenize_refcoco)
    tokenized_val_dataset = LazyRefCOCODataset(val_dataset, tokenize_refcoco)
    tokenized_test_dataset = LazyRefCOCODataset(test_dataset, tokenize_refcoco)

    train_len = len(tokenized_train_dataset)
    val_len = len(tokenized_val_dataset)
    test_len = len(tokenized_test_dataset)

    if train_len == 0:
        raise ValueError(
            "No training samples after processing. "
            "Possible causes: images not loading correctly, dataset format incompatible, "
            "or all examples failed validation."
        )

    # Fail fast: tokenize one example now (also prints the debug block above).
    _probe = tokenized_train_dataset[0]
    assert _probe is not None

    print(f"\nSuccessfully prepared datasets:")
    print(f"Training samples: {train_len}")
    print(f"Validation samples: {val_len}")
    print(f"Test samples: {test_len}")

    eff_batch = config.batch_size * config.gradient_accumulation_steps
    total_steps = math.ceil(train_len / eff_batch) * config.num_epochs

    print(f"\nTraining Configuration:")
    print(f"Total training steps: {total_steps}")
    print(f"Effective batch size: {eff_batch}")
    print(f"Steps per epoch: {math.ceil(train_len / eff_batch)}")

    training_args = Seq2SeqTrainingArguments(
        output_dir=os.path.join(OUTPUT_DIR, f"results_refcoco_lora_llama11b_seed{SEED}"),
        per_device_train_batch_size=config.batch_size,
        per_device_eval_batch_size=config.batch_size,
        gradient_accumulation_steps=config.gradient_accumulation_steps,
        num_train_epochs=config.num_epochs,
        learning_rate=config.learning_rate,
        bf16=(config.precision == "bf16"),
        fp16=(config.precision == "fp16"),
        max_grad_norm=1.5,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        dataloader_num_workers=config.num_workers,
        dataloader_pin_memory=config.pin_memory,
        dataloader_drop_last=config.drop_last,
        remove_unused_columns=False,
        report_to="none",

        save_strategy="no",
        eval_strategy="no",
        load_best_model_at_end=False,

        logging_steps=config.logging_steps,
        warmup_steps=config.warmup_steps,

        optim="adamw_torch",
        prediction_loss_only=False,
        predict_with_generate=True,

        seed=SEED,
    )

    pixel_dtype = torch.bfloat16 if config.precision == "bf16" else (
        torch.float16 if config.precision == "fp16" else torch.float32
    )

    def _pad_tiles(t, target_tiles, tile_dim):
        cur = t.shape[tile_dim]
        if cur == target_tiles:
            return t
        pad_shape = list(t.shape)
        pad_shape[tile_dim] = target_tiles - cur
        return torch.cat([t, t.new_zeros(pad_shape)], dim=tile_dim)

    def collate_fn(batch):
        """
        Mllama collator.
        - RIGHT-pads input_ids / attention_mask / labels (so prediction_step
          can take the prompt as input_ids[i, :prompt_length]).
        - Pads cross_attention_mask along the sequence dim with zeros so it
          always has exactly the same length as input_ids.
        - Stacks the per-image vision tensors.
        """
        batch = [b for b in batch if b is not None and isinstance(b, dict) and 'input_ids' in b]
        if len(batch) == 0:
            raise RuntimeError("Empty batch in collate_fn.")

        pad_id = tokenizer.pad_token_id

        input_ids = torch.nn.utils.rnn.pad_sequence(
            [b['input_ids'] for b in batch], batch_first=True, padding_value=pad_id
        )
        attention_mask = torch.nn.utils.rnn.pad_sequence(
            [b['attention_mask'] for b in batch], batch_first=True, padding_value=0
        )
        labels = torch.nn.utils.rnn.pad_sequence(
            [b['labels'] for b in batch], batch_first=True, padding_value=config.ignore_index
        )
        max_len = input_ids.shape[1]

        # The processor pads to max_image_tiles already; this is just a guard.
        max_tiles = max(b['pixel_values'].shape[1] for b in batch)

        cams, pvs, ar_ids, ar_masks = [], [], [], []
        for b in batch:
            cam = _pad_tiles(b['cross_attention_mask'], max_tiles, tile_dim=-1)
            seq_pad = max_len - cam.shape[0]
            if seq_pad > 0:
                cam = torch.cat([cam, cam.new_zeros((seq_pad,) + tuple(cam.shape[1:]))], dim=0)
            cams.append(cam)
            pvs.append(_pad_tiles(b['pixel_values'], max_tiles, tile_dim=1))
            ar_ids.append(b['aspect_ratio_ids'])
            ar_masks.append(_pad_tiles(b['aspect_ratio_mask'], max_tiles, tile_dim=-1))

        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': labels,
            'cross_attention_mask': torch.stack(cams, dim=0),          # [B, seq, n_img, n_tiles]
            'pixel_values': torch.stack(pvs, dim=0).to(pixel_dtype),  # [B, n_img, n_tiles, 3, H, W]
            'aspect_ratio_ids': torch.stack(ar_ids, dim=0),            # [B, n_img]
            'aspect_ratio_mask': torch.stack(ar_masks, dim=0),         # [B, n_img, n_tiles]
            # metadata — popped / stripped before the model sees the batch
            'refcoco_answers': [b.get('refcoco_answers', []) for b in batch],
            'prompt_length': torch.tensor([int(b['prompt_length']) for b in batch], dtype=torch.long),
        }

    def make_compute_metrics(processor, eval_answers_buffer):
        import re
        import numpy as np
        from rouge_score import rouge_scorer
        from nltk.translate.meteor_score import meteor_score
        from bert_score import score as bert_score
        from collections import defaultdict
        import nltk
        nltk.download('wordnet', quiet=True)
        nltk.download('omw-1.4', quiet=True)

        # PATCH FOR MOVERSCORE COMPATIBILITY
        import transformers.tokenization_utils_base as tub
        if not hasattr(tub.PreTrainedTokenizerBase, '_moverscore_patched'):
            original_init = tub.PreTrainedTokenizerBase.__init__
            def patched_init(self, *args, **kwargs):
                original_init(self, *args, **kwargs)
                if not hasattr(self, 'max_len'):
                    self.max_len = self.model_max_length
            tub.PreTrainedTokenizerBase.__init__ = patched_init
            tub.PreTrainedTokenizerBase._moverscore_patched = True
            print("✓ Applied moverscore compatibility patch")

        # Now import moverscore
        moverscore_available = False
        try:
            from moverscore_v2 import get_idf_dict, word_mover_score
            moverscore_available = True
            print("✓ MoverScore loaded successfully")
        except ImportError as e:
            print(f"✗ MoverScore not available: {e}")

        scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)

        tok = processor.tokenizer

        def extract_answer(text):
            """Clean Llama-3 output safely without truncating content."""

            if text is None:
                return ""

            # Remove Llama-3 special tokens (normally already skipped by decode)
            text = text.replace("<|begin_of_text|>", "")
            text = text.replace("<|end_of_text|>", "")
            text = text.replace("<|start_header_id|>", "")
            text = text.replace("<|end_header_id|>", "")
            text = text.replace("<|eot_id|>", "")
            text = text.replace("<|image|>", "")

            # Remove an accidental leading assistant header if it survives decoding
            text = re.sub(r"^\s*assistant\s*", "", text, flags=re.IGNORECASE)

            # Normalize whitespace
            text = re.sub(r"\s+", " ", text).strip()

            return text

        # Official VQA eval answer processing (from the VQA project's
        # vqaEval.py), used so exact-match comparisons for accuracy (and
        # the text-similarity metrics below) aren't broken by trailing
        # punctuation, articles, or number-word vs. digit mismatches.
        _vqa_contractions = {
            "aint": "ain't", "arent": "aren't", "cant": "can't", "couldve": "could've",
            "couldnt": "couldn't", "didnt": "didn't", "doesnt": "doesn't", "dont": "don't",
            "hadnt": "hadn't", "hasnt": "hasn't", "havent": "haven't", "hed": "he'd",
            "hes": "he's", "im": "i'm", "ive": "i've", "isnt": "isn't", "its": "it's",
            "lets": "let's", "shant": "shan't", "shes": "she's", "shouldve": "should've",
            "shouldnt": "shouldn't", "thats": "that's", "theres": "there's", "theyd": "they'd",
            "theyre": "they're", "theyve": "they've", "wasnt": "wasn't", "wed": "we'd",
            "were": "we're", "weve": "we've", "werent": "weren't", "whats": "what's",
            "wheres": "where's", "whos": "who's", "wont": "won't", "wouldve": "would've",
            "wouldnt": "wouldn't", "youd": "you'd", "youre": "you're", "youve": "you've",
        }
        _vqa_number_words = {
            "none": "0", "zero": "0", "one": "1", "two": "2", "three": "3", "four": "4",
            "five": "5", "six": "6", "seven": "7", "eight": "8", "nine": "9", "ten": "10",
        }
        _vqa_articles = {"a", "an", "the"}
        _vqa_punct = [';', '/', '[', ']', '"', '{', '}', '(', ')', '=', '+', '\\',
                      '_', '-', '>', '<', '@', '`', ',', '?', '!', '.', ':']

        def normalize(s):
            """Official VQA-style answer normalization."""
            s = s.lower().strip()
            # Remove punctuation entirely (rather than spacing it out),
            # so "brown." normalizes to "brown", not "brown .".
            for p in _vqa_punct:
                s = s.replace(p, ' ')
            s = re.sub(r"\s+", " ", s).strip()

            words = []
            for word in s.split():
                word = _vqa_number_words.get(word, word)
                if word in _vqa_articles:
                    continue
                word = _vqa_contractions.get(word, word)
                words.append(word)
            return ' '.join(words).strip()

        def safe_ids(arr):
            """
            Llama-safe id cleanup before decoding (predictions AND labels).
            - Any negative id (-100 label mask, or the -100 the Trainer uses to
              pad generations across eval batches) -> pad id. Clipping -100 to
              0 instead would decode as "!" in Llama's vocab.
            - Upper bound is len(tokenizer) (128k base + added special tokens),
              NOT tokenizer.vocab_size (base only): clipping to vocab_size-1
              would turn <|eot_id|> (128009) into a real text token (127999)
              that skip_special_tokens can no longer remove.
            """
            arr = np.asarray(arr, dtype=np.int64).copy()
            pad_token_id = tok.pad_token_id
            arr[arr < 0] = pad_token_id
            arr[arr >= len(tok)] = pad_token_id
            return arr

        def compute_refcoco_metrics(p: EvalPrediction):
            pred_ids = safe_ids(p.predictions)
            labels = safe_ids(p.label_ids)

            try:
                pred_texts_raw = tok.batch_decode(pred_ids, skip_special_tokens=True)
                label_texts = tok.batch_decode(labels, skip_special_tokens=True)
            except Exception as e:
                print(f"[Metrics] Decode error: {e}")
                return {"rouge_l": 0.0, "meteor": 0.0, "bertscore_f1": 0.0, "moverscore": 0.0, "accuracy": 0.0}

            pred_texts = [extract_answer(t) for t in pred_texts_raw]
            pred_norm = [normalize(t) for t in pred_texts]
            label_texts = [extract_answer(t) for t in label_texts]
            label_norm = [normalize(t) for t in label_texts]

            for i in range(min(3, len(pred_norm))):
                print(f"\n[Sample {i+1}]")
                print(f"Generated: {pred_texts[i][:100]}")
                print(f"Reference: {label_texts[i][:100]}")

            # FIX (root of the "half-baked scores" issue): everything below
            # used to be scored against `label_norm`, which is decoded from
            # the `labels` tensor — and `labels` only ever encodes
            # answers_list[0] (see tokenize_refcoco), i.e. ONE arbitrary human
            # reference out of the 2-4 RefCOCO collects per region. A
            # correct prediction that happens to match reference #2 or #3
            # instead of #1 scored as a near-total miss on ROUGE-L/METEOR.
            #
            # lmms-eval's own aggregation for this exact dataset
            # (refcoco_aggregation_result) instead feeds ALL of
            # doc["answer"] in as separate reference captions for the same
            # image_id and lets pycocoevalcap's scorers take the best
            # match across the full reference set. We reproduce that here
            # using eval_answers_buffer (already populated by
            # LoRATrainer.prediction_step with the raw 2-4 answers per
            # example) instead of the single-answer label_norm.
            raw_answers_list = eval_answers_buffer.get("answers", [])
            use_multi_ref = len(raw_answers_list) == len(pred_norm)

            if use_multi_ref:
                norm_refs_per_example = [
                    [normalize(a) for a in raw_answers if a] or [l]
                    for raw_answers, l in zip(raw_answers_list, label_norm)
                ]
            else:
                print(
                    f"[Metrics] WARNING: buffered answers ({len(raw_answers_list)}) "
                    f"!= predictions ({len(pred_norm)}); falling back to "
                    f"single-reference scoring against the training answer."
                )
                norm_refs_per_example = [[l] for l in label_norm]

            valid_pairs = [
                (p, refs) for p, refs in zip(pred_norm, norm_refs_per_example)
                if p and refs and len(p.split()) > 0
            ]

            # ROUGE-L — max over this example's reference set.
            rouge_ls = []
            for p, refs in valid_pairs:
                best = max(
                    (scorer.score(r, p)['rougeL'].fmeasure for r in refs if r),
                    default=None,
                )
                if best is not None:
                    rouge_ls.append(best)
            rouge_l = np.mean(rouge_ls) if rouge_ls else 0.0

            # METEOR — nltk's meteor_score natively supports multiple
            # references and takes the max internally, so just pass the
            # full reference set instead of a single one.
            meteors = []
            for p, refs in valid_pairs:
                ref_token_lists = [r.split() for r in refs if r]
                if ref_token_lists:
                    meteors.append(meteor_score(ref_token_lists, p.split()))
            meteor = np.mean(meteors) if meteors else 0.0

            # BERTscore / MoverScore need one (pred, ref) string pair per
            # call, so we flatten pred-vs-each-of-its-refs into parallel
            # lists, score everything in one batched call, then take the
            # max back per original example.
            flat_preds, flat_refs, group_idx = [], [], []
            for i, (p, refs) in enumerate(valid_pairs):
                for r in refs:
                    if r:
                        flat_preds.append(p)
                        flat_refs.append(r)
                        group_idx.append(i)

            def _max_per_group(flat_scores, n_groups):
                best = [None] * n_groups
                for idx, s in zip(group_idx, flat_scores):
                    if best[idx] is None or s > best[idx]:
                        best[idx] = s
                return [b for b in best if b is not None]

            # BERTscore
            if flat_preds and flat_refs:
                try:
                    P, R, F1 = bert_score(
                        flat_preds,
                        flat_refs,
                        lang="en",
                        model_type="bert-base-uncased",
                        rescale_with_baseline=False
                    )
                    per_ex_best = _max_per_group(F1.tolist(), len(valid_pairs))
                    bertscore_f1 = float(np.mean(per_ex_best)) if per_ex_best else 0.0
                except Exception as e:
                    print(f"[Metrics] BERTScore error: {e}")
                    bertscore_f1 = 0.0
            else:
                bertscore_f1 = 0.0

            # Accuracy — FIX: the original VQA leave-one-out soft-accuracy
            # formula is designed for VQA's setting (short, closed-set
            # answers like "umpire", ~10 annotators) and doesn't transfer
            # well to RefCOCO's setting (free-form referring expressions,
            # only 2-4 annotators):
            #   1) The "/3" threshold caps max achievable accuracy below 1.0
            #      even for a perfect prediction with only 2-4 references
            #      (e.g. ceiling ≈ 0.67 with 3 refs).
            #   2) It requires an EXACT string match. A correct caption like
            #      "this is an umpire" will never exactly equal the
            #      reference "umpire", so accuracy stays near 0 regardless
            #      of how good the model actually is. This is why standard
            #      RefCOCO generation papers report METEOR/CIDEr/ROUGE-L/
            #      BLEU rather than "accuracy" at all — free-form text
            #      doesn't have a single correct string to match against.
            #
            # Fix: use a relaxed containment match instead — a prediction
            # counts as correct for an example if it exactly matches, or
            # contains, or is contained by, any one of that example's human
            # reference expressions (after normalization). This still
            # requires real word-level overlap with a genuine reference (so
            # it isn't free credit), but no longer structurally punishes a
            # correct model for writing "this is an umpire" instead of
            # exactly "umpire", and isn't capped by a VQA-specific ceiling.
            # Treat this as a coarse correct/incorrect signal — ROUGE-L /
            # METEOR / BERTScore / MoverScore remain the primary metrics
            # for judging caption quality.
            # (raw_answers_list was already fetched above for the
            # multi-reference NLG metrics — reused here as-is.)

            print(f"\n[Accuracy Debug] buffered answer-sets: {len(raw_answers_list)} | predictions: {len(pred_norm)}")
            for i in range(min(3, len(raw_answers_list), len(pred_norm))):
                print(f"[Accuracy Debug] Sample {i+1}")
                print(f"  Prediction (normalized): {pred_norm[i]!r}")
                print(f"  Raw human answers       : {raw_answers_list[i]}")
                print(f"  Normalized human answers: {[normalize(a) for a in raw_answers_list[i] if a]}")

            def relaxed_match(pred, ref):
                if not pred or not ref:
                    return False
                if pred == ref:
                    return True
                # Require at least a few characters of real overlap so a
                # single short common word (e.g. "the") can't trivially
                # match — both strings are already normalized (articles
                # stripped, lowercased) at this point.
                if len(ref) >= 3 and ref in pred:
                    return True
                if len(pred) >= 3 and pred in ref:
                    return True
                return False

            if not use_multi_ref:
                # Warning already printed above when building
                # norm_refs_per_example — just apply the same fallback here.
                matches = [1.0 if relaxed_match(p, l) else 0.0 for p, l in zip(pred_norm, label_norm) if l]
                accuracy = float(np.mean(matches)) if matches else 0.0
            else:
                per_question_accs = []
                for pred, raw_answers in zip(pred_norm, raw_answers_list):
                    norm_answers = [normalize(a) for a in raw_answers if a]
                    if not norm_answers:
                        continue
                    is_correct = any(relaxed_match(pred, ref) for ref in norm_answers)
                    per_question_accs.append(1.0 if is_correct else 0.0)
                accuracy = float(np.mean(per_question_accs)) if per_question_accs else 0.0

            # MoverScore — same flatten-then-max-per-example approach as
            # BERTScore above, using flat_preds/flat_refs/group_idx built
            # earlier from the FULL reference set (not just answers_list[0]).
            moverscore_val = 0.0
            if moverscore_available and flat_preds and flat_refs:
                try:
                    from moverscore_v2 import get_idf_dict, word_mover_score

                    idf_dict_ref = get_idf_dict(flat_refs)
                    idf_dict_hyp = get_idf_dict(flat_preds)

                    scores = word_mover_score(
                        flat_refs,
                        flat_preds,
                        idf_dict_ref,
                        idf_dict_hyp,
                        stop_words=[],
                        n_gram=1,
                        remove_subwords=True
                    )

                    if scores and len(scores) > 0:
                        per_ex_best = _max_per_group(scores, len(valid_pairs))
                        moverscore_val = float(np.mean(per_ex_best)) if per_ex_best else 0.0
                        print(f"[MoverScore] Score: {moverscore_val:.4f}")

                except Exception as e:
                    print(f"[MoverScore] Error: {e}")
                    import traceback
                    traceback.print_exc()

            return {
                "rouge_l": float(rouge_l),
                "meteor": float(meteor),
                "bertscore_f1": float(bertscore_f1),
                "moverscore": float(moverscore_val),
                "accuracy": accuracy,
            }

        return compute_refcoco_metrics

    class LoRATrainer(Seq2SeqTrainer):
        """Plain LoRA Seq2SeqTrainer (no GRIT) with Llama-safe metadata handling and generation."""

        def __init__(self, processor, eval_answers_buffer, gen_config, eos_token_ids, *args, **kwargs):
            super().__init__(*args, **kwargs)
            self.processor = processor
            # Mutable dict shared with make_compute_metrics(): holds the raw 2-4
            # human referring expressions per eval example for multi-reference scoring.
            self.eval_answers_buffer = eval_answers_buffer
            self.gen_config = gen_config  # LoRAConfig instance
            # Llama-3 instruct turns end with <|eot_id|>, not the base EOS.
            self.eos_token_ids = eos_token_ids

        def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
            # Standard Trainer loss — only real model inputs go to Mllama's
            # forward (Gemma silently ignored the metadata keys; Mllama doesn't).
            model_inputs = {k: v for k, v in inputs.items() if k not in NON_MODEL_KEYS}
            return super().compute_loss(
                model, model_inputs, return_outputs=return_outputs, num_items_in_batch=num_items_in_batch
            )

        def evaluate(self, *args, **kwargs):
            print("\nClearing VRAM before evaluation...")
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            self.eval_answers_buffer["answers"] = []
            return super().evaluate(*args, **kwargs)

        def prediction_step(
            self,
            model: torch.nn.Module,
            inputs: Dict[str, torch.Tensor],
            prediction_loss_only: bool,
            ignore_keys: Optional[list] = None,
        ) -> Tuple[Optional[torch.Tensor], Optional[torch.Tensor], Optional[torch.Tensor]]:
            """
            Llama / Mllama version.

            inputs["input_ids"] is the FULL chat-templated sequence
                <|begin_of_text|> user header <|image|> instruction <|eot_id|>
                assistant header  answer <|eot_id|>
            produced by ONE processor call, so cross_attention_mask matches it
            position-for-position. The collator RIGHT-pads, so the prompt of
            example i is always input_ids[i, :prompt_length[i]].

            For generation we SLICE (never extend) input_ids, attention_mask AND
            cross_attention_mask to the prompt length, then left-repad. Slicing a
            processor-built mask gives exactly what a prompt-only processor call
            would give (tokenize_refcoco asserts the prompt ids are an exact
            prefix of the full ids), so the text/mask lengths always agree ->
            no device-side assert. pixel_values / aspect_ratio_ids /
            aspect_ratio_mask are per-image and carried over unchanged.

            The teacher-forced eval loss still uses the full [prompt][answer]
            sequence.
            """
            labels = inputs.pop("labels", None)
            has_labels = labels is not None

            # Raw human answers: plain Python lists, never sent to the model.
            ref_answers = inputs.pop("refcoco_answers", None)
            if ref_answers is not None:
                self.eval_answers_buffer["answers"].extend(ref_answers)

            prompt_lengths = inputs.pop("prompt_length", None)

            inputs = self._prepare_inputs(inputs)
            input_ids = inputs["input_ids"]
            input_len = input_ids.shape[1]

            if prompt_lengths is not None:
                attn = inputs["attention_mask"]
                cam = inputs.get("cross_attention_mask", None)
                pad_id = self.processor.tokenizer.pad_token_id
                bsz = input_ids.shape[0]
                p_lens = [int(prompt_lengths[i]) for i in range(bsz)]
                max_p = max(p_lens)

                gen_input_ids = torch.full((bsz, max_p), pad_id, dtype=input_ids.dtype, device=input_ids.device)
                gen_attention_mask = torch.zeros((bsz, max_p), dtype=attn.dtype, device=attn.device)
                gen_cam = None
                if cam is not None:
                    gen_cam = torch.zeros((bsz, max_p) + tuple(cam.shape[2:]), dtype=cam.dtype, device=cam.device)

                # Re-pad on the LEFT for batched decoding.
                for i, p in enumerate(p_lens):
                    gen_input_ids[i, max_p - p:] = input_ids[i, :p]
                    gen_attention_mask[i, max_p - p:] = attn[i, :p]
                    if gen_cam is not None:
                        gen_cam[i, max_p - p:] = cam[i, :p]

                gen_inputs = {
                    "input_ids": gen_input_ids,
                    "attention_mask": gen_attention_mask,
                }
                if gen_cam is not None:
                    gen_inputs["cross_attention_mask"] = gen_cam
                for k in ("pixel_values", "aspect_ratio_ids", "aspect_ratio_mask"):
                    if k in inputs:
                        gen_inputs[k] = inputs[k]

                gen_input_len = gen_input_ids.shape[1]
            else:
                # Should not happen with tokenize_refcoco; kept as a fallback.
                gen_inputs = {k: v for k, v in inputs.items() if k not in NON_MODEL_KEYS}
                gen_input_len = input_len

            with torch.no_grad():
                if prediction_loss_only:
                    loss = None
                    if has_labels:
                        inputs_copy = dict(inputs)
                        inputs_copy["labels"] = labels
                        loss = self.compute_loss(model, inputs_copy).detach()
                    return (loss, None, None)

                generated_tokens = model.generate(
                    **gen_inputs,
                    max_new_tokens=self.gen_config.generation_max_new_tokens,
                    min_new_tokens=2,
                    do_sample=False,
                    use_cache=True,
                    pad_token_id=self.processor.tokenizer.pad_token_id,
                    eos_token_id=self.eos_token_ids,
                )

                # Decoder-only output = [prompt][new tokens]; keep new tokens only.
                generated_tokens = generated_tokens[:, gen_input_len:]

                # Teacher-forced loss on the FULL [prompt][answer] sequence.
                loss = None
                if has_labels:
                    inputs_copy = dict(inputs)
                    inputs_copy["labels"] = labels
                    loss = self.compute_loss(model, inputs_copy).detach()

            if labels is not None:
                if labels.dim() == 3:
                    labels = torch.argmax(labels, dim=-1)
                elif labels.dim() == 1:
                    batch_size = input_ids.shape[0]
                    labels = labels.view(batch_size, -1)

            return loss, generated_tokens, labels


    # Shared mutable buffer: LoRATrainer.prediction_step fills it with the
    # raw per-question human answers for each eval batch; compute_metrics
    # reads it to compute the multi-reference NLG metrics / relaxed-match accuracy.
    eval_answers_buffer = {"answers": []}

    trainer = LoRATrainer(
        processor=processor,
        eval_answers_buffer=eval_answers_buffer,
        gen_config=config,
        eos_token_ids=eos_token_ids,
        model=model,
        args=training_args,
        train_dataset=tokenized_train_dataset,
        eval_dataset=tokenized_val_dataset,
        data_collator=collate_fn,
        compute_metrics=make_compute_metrics(processor, eval_answers_buffer),
    )

    print("\n" + "="*80)
    print("STARTING LORA-ONLY TRAINING FOR LLAMA-3.2-11B-VISION-INSTRUCT ON REFCOCO")
    print("="*80)
    print(f"Model: {config.model_id}")
    print(f"Dataset: {config.dataset_name}")
    print(f"LoRA rank: {config.lora_rank}")
    print(f"Batch size: {config.batch_size} x {config.gradient_accumulation_steps} = {eff_batch}")
    print(f"Learning rate: {config.learning_rate}")
    print(f"Precision: {config.precision}")
    print(f"Training samples: {train_len}")
    print(f"Validation samples: {val_len}")
    print(f"Test samples: {test_len}")
    print(f"Mode: {'SMOKE TEST' if SMOKE_TEST else 'FULL'}")
    print(f"Output dir: {OUTPUT_DIR}")
    print("="*80 + "\n")

    try:
        print("Beginning training...")
        trainer.train()

        print("\n" + "="*80)
        print("FINAL TEST EVALUATION")
        print("="*80)

        test_results = trainer.evaluate(eval_dataset=tokenized_test_dataset)

        print(f"Test Loss          : {test_results['eval_loss']:.5f}")
        print(f"Test ROUGE-L       : {test_results['eval_rouge_l']:.4f}")
        print(f"Test METEOR        : {test_results['eval_meteor']:.4f}")
        print(f"Test BERTscore F1  : {test_results['eval_bertscore_f1']:.4f}")
        print(f"Test MoverScore    : {test_results['eval_moverscore']:.4f}")
        print(f"Test Accuracy      : {test_results['eval_accuracy']:.4f}")

        final_results = {
            "method": "LoRA",
            "model": config.model_id,
            "task": "RefCOCO",
            "seed": SEED,
            "smoke_test": SMOKE_TEST,
            "lora_rank": config.lora_rank,
            "lora_alpha": config.lora_alpha,
            "lora_dropout": config.lora_dropout,
            "language_target_modules": config.lora_target_modules,
            "vision_target_modules": config.vision_lora_modules,
            "lora_target_regex": target_regex,
            "learning_rate": config.learning_rate,
            "epochs": config.num_epochs,
            "effective_batch_size": eff_batch,
            "optimizer_steps": total_steps,
            "train_samples": train_len,
            "val_samples": val_len,
            "test_samples": test_len,
            "test_metrics": {
                "loss": float(test_results["eval_loss"]),
                "rouge_l": float(test_results["eval_rouge_l"]),
                "meteor": float(test_results["eval_meteor"]),
                "bertscore_f1": float(test_results["eval_bertscore_f1"]),
                "moverscore": float(test_results["eval_moverscore"]),
                "accuracy": float(test_results["eval_accuracy"]),
            }
        }

        results_path = os.path.join(OUTPUT_DIR, f"{RUN_NAME}.json")

        with open(results_path, "w") as f:
            json.dump(final_results, f, indent=2)

        print(f"Results saved to: {results_path}")

        print("\n" + "="*80)
        print("Saving final model...")
        output_dir = os.path.join(OUTPUT_DIR, "final_refcoco_lora_llama11b_model")
        model.save_pretrained(output_dir)
        processor.save_pretrained(output_dir)
        print(f"Model saved to {output_dir}")

        print("\n" + "="*80)
        print("TRAINING COMPLETED SUCCESSFULLY")
        print("="*80 + "\n")

    except KeyboardInterrupt:
        print("\n" + "="*80)
        print("TRAINING INTERRUPTED BY USER")
        print("="*80)
        print("Saving checkpoint...")
        try:
            ckpt_dir = os.path.join(OUTPUT_DIR, "interrupted_checkpoint_lora_llama11b")
            model.save_pretrained(ckpt_dir)
            processor.save_pretrained(ckpt_dir)
            print(f"Checkpoint saved to {ckpt_dir}")
        except Exception as e:
            print(f"Failed to save checkpoint: {e}")

    except Exception as e:
        print("\n" + "="*80)
        print("TRAINING FAILED")
        print("="*80)
        print(f"Error: {e}")
        import traceback
        traceback.print_exc()
        raise

    finally:
        print("\nCleaning up...")
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        print("Cleanup complete")


if __name__ == "__main__":
    main()


All experiment outputs will be saved in:
/LoRA_llama11b_seed42_RefCOCO_smoke
Loading processor and model for Llama-3.2-11B-Vision-Instruct (RefCOCO)...
Pad token: '<|finetune_right_pad_id|>' (id=128004)
EOS token ids for generation: [128009, 128001]


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/906 [00:00<?, ?it/s]

LoRA target regex: (?:.*language_model\..*\.(q_proj|k_proj|v_proj))|(?:.*vision_model\..*\.(q_proj|k_proj|v_proj))
✓ Model has no meta parameters
✓ Model device: cuda:0
Instrumented 120 vision modules + 120 language modules
Total LoRA modules: 240
Trainable params: 4,177,920
Total params: 10,674,398,755
Percentage: 0.039%
Loading RefCOCO dataset with splits...
Final splits → Train: 64, Val: 8, Test: 8
Building lazy datasets (on-the-fly tokenization)...

[Tokenization Debug] ---------------------------------
Full seq len: 41 | prompt len: 38 | target len: 3
Prompt tail : ' extra text.<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n'
Target (raw): 'red shirt<|eot_id|>'
pixel_values (1, 4, 3, 560, 560) | aspect_ratio_ids (1,) | aspect_ratio_mask (1, 4) | cross_attention_mask (41, 1, 4)
-----------------------------------------------------


Successfully prepared datasets:
Training samples: 64
Validation samples: 8
Test samples: 8

Training Configuration:
Total training steps: 4

/venv/main/lib/python3.12/site-packages/torch/nn/modules/module.py:1790: FutureWarning: `hidden_state` is deprecated and will be removed in version v5.20 for `MllamaVisionEncoderLayer.forward`. Use `hidden_states` instead.
  return forward_call(*args, **kwargs)


Step,Training Loss



TRAINING INTERRUPTED BY USER
Saving checkpoint...
Checkpoint saved to /LoRA_llama11b_seed42_RefCOCO_smoke/interrupted_checkpoint_lora_llama11b

Cleaning up...
Cleanup complete
